# InstaSHAP Extension: Interaction-Aware InstaSHAP

## DS357 - Explainable AI Course Project

This notebook demonstrates our extension to the InstaSHAP paper that addresses the limitation of purely additive surrogates.

### Research Gap

InstaSHAP relies on additive GAM surrogates that cannot capture feature interactions. When interactions are significant, the additive surrogate poorly approximates the black-box model, leading to inaccurate Shapley values.

### Our Solution

We propose **Interaction-Aware InstaSHAP** using GA²M surrogates with pairwise interaction terms.

In [ ]:
# Setup and imports
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

# Add paths
sys.path.append('..')
sys.path.append('../../phase2')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Setup complete!")

## 1. Demonstrating the Gap

First, let's create a synthetic dataset with known feature interactions and show that the original InstaSHAP struggles.

In [ ]:
from phase2.data.data_loader import create_synthetic_interaction_dataset
from phase2.models.base_model import train_model_for_dataset

# Create dataset with STRONG interactions
strong_int_data = create_synthetic_interaction_dataset(
    n_samples=2000,
    n_features=10,
    interaction_strength=2.0,  # Strong interaction
    noise_level=0.1
)

# Train black-box model
blackbox = train_model_for_dataset(strong_int_data, model_type='xgboost')

print("\nDataset: Synthetic with strong feature interactions")
print(f"The target includes: x0 * x1 interaction term (strength=2.0)")

In [ ]:
from phase2.models.gam_surrogate import train_surrogate_for_blackbox
from phase2.models.instashap import InstaSHAP

# Train ADDITIVE surrogate (original InstaSHAP approach)
additive_surrogate = train_surrogate_for_blackbox(
    blackbox.model,
    strong_int_data['X_train'],
    task_type='regression'
)

# Check fidelity
bb_preds = blackbox.model.predict(strong_int_data['X_test'])
add_preds = additive_surrogate.predict(strong_int_data['X_test'])
add_r2 = r2_score(bb_preds, add_preds)

print(f"\nAdditive Surrogate Fidelity: R² = {add_r2:.4f}")
print("Note: Low fidelity indicates the additive model cannot capture the interaction!")

## 2. Our Extension: Interaction-Aware Surrogate

Now let's use our GA²M surrogate that includes pairwise interactions.

In [ ]:
from extension.interaction_aware_surrogate import train_interaction_surrogate_for_blackbox

# Train GA²M surrogate with interactions
ga2m_surrogate = train_interaction_surrogate_for_blackbox(
    blackbox.model,
    strong_int_data['X_train'],
    task_type='regression',
    n_interactions=5
)

# Check fidelity
ga2m_preds = ga2m_surrogate.predict(strong_int_data['X_test'])
ga2m_r2 = r2_score(bb_preds, ga2m_preds)

print(f"\nGA²M Surrogate Fidelity: R² = {ga2m_r2:.4f}")
print(f"Improvement over additive: {ga2m_r2 - add_r2:.4f}")
print(f"\nDetected interaction pairs: {ga2m_surrogate.get_interaction_pairs()}")

In [ ]:
# Visualize fidelity improvement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Additive surrogate
ax1 = axes[0]
ax1.scatter(bb_preds, add_preds, alpha=0.3, s=10)
ax1.plot([bb_preds.min(), bb_preds.max()], [bb_preds.min(), bb_preds.max()], 'r--', lw=2)
ax1.set_xlabel('Black-Box Predictions')
ax1.set_ylabel('Surrogate Predictions')
ax1.set_title(f'Additive GAM (R² = {add_r2:.4f})')

# GA²M surrogate
ax2 = axes[1]
ax2.scatter(bb_preds, ga2m_preds, alpha=0.3, s=10)
ax2.plot([bb_preds.min(), bb_preds.max()], [bb_preds.min(), bb_preds.max()], 'r--', lw=2)
ax2.set_xlabel('Black-Box Predictions')
ax2.set_ylabel('Surrogate Predictions')
ax2.set_title(f'GA²M with Interactions (R² = {ga2m_r2:.4f})')

plt.tight_layout()
plt.show()

## 3. Enhanced InstaSHAP Values

Now let's compute Shapley values using both methods and compare with Exact SHAP.

In [ ]:
from phase2.explainers.exact_shap import ExactSHAPExplainer
from extension.enhanced_instashap import EnhancedInstaSHAP

X_explain = strong_int_data['X_test'].iloc[:100]

# Compute Exact SHAP (ground truth)
exact_explainer = ExactSHAPExplainer(
    blackbox.model,
    strong_int_data['X_train'],
    model_type='tree',
    task_type='regression'
)
exact_shap = exact_explainer.explain(X_explain, return_dataframe=False)

# Compute Original InstaSHAP
original_instashap = InstaSHAP(additive_surrogate, strong_int_data['X_train'])
original_shap = original_instashap.explain(X_explain, return_dataframe=False)

# Compute Enhanced InstaSHAP
enhanced_instashap = EnhancedInstaSHAP(ga2m_surrogate, strong_int_data['X_train'])
enhanced_shap = enhanced_instashap.explain(X_explain, return_dataframe=False)

print("SHAP values computed!")

In [ ]:
# Compare accuracy
exact_flat = exact_shap.flatten()
original_flat = original_shap.flatten()
enhanced_flat = enhanced_shap.flatten()

original_r, _ = pearsonr(exact_flat, original_flat)
enhanced_r, _ = pearsonr(exact_flat, enhanced_flat)

print(f"Correlation with Exact SHAP:")
print(f"  Original InstaSHAP: r = {original_r:.4f}")
print(f"  Enhanced InstaSHAP: r = {enhanced_r:.4f}")
print(f"  Improvement: {enhanced_r - original_r:.4f}")

In [ ]:
# Visualize accuracy comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Original InstaSHAP vs Exact
ax1 = axes[0]
ax1.scatter(exact_flat, original_flat, alpha=0.3, s=10)
ax1.plot([exact_flat.min(), exact_flat.max()], 
         [exact_flat.min(), exact_flat.max()], 'r--', lw=2)
ax1.set_xlabel('Exact SHAP')
ax1.set_ylabel('Original InstaSHAP')
ax1.set_title(f'Original InstaSHAP (r = {original_r:.4f})')

# Enhanced InstaSHAP vs Exact
ax2 = axes[1]
ax2.scatter(exact_flat, enhanced_flat, alpha=0.3, s=10)
ax2.plot([exact_flat.min(), exact_flat.max()], 
         [exact_flat.min(), exact_flat.max()], 'r--', lw=2)
ax2.set_xlabel('Exact SHAP')
ax2.set_ylabel('Enhanced InstaSHAP')
ax2.set_title(f'Enhanced InstaSHAP (r = {enhanced_r:.4f})')

plt.tight_layout()
plt.show()

## 4. Interaction Importance

The Enhanced InstaSHAP can also report the importance of detected interactions.

In [ ]:
# Get interaction importance
int_importance = enhanced_instashap.get_interaction_importance(X_explain)

print("Interaction Importance:")
for pair, imp in sorted(int_importance.items(), key=lambda x: x[1], reverse=True):
    feat_names = [strong_int_data['feature_names'][i] for i in pair]
    print(f"  {feat_names[0]} × {feat_names[1]}: {imp:.4f}")

## 5. Adaptive Strategy

Our adaptive surrogate automatically selects between additive GAM and GA²M based on fidelity.

In [ ]:
from extension.adaptive_surrogate import AdaptiveSurrogate

# Test on interaction-heavy data
print("Test 1: Dataset with STRONG interactions")
adaptive1 = AdaptiveSurrogate(fidelity_threshold=0.85)
adaptive1.fit(blackbox.model, strong_int_data['X_train'])

print(f"\nDecision: {adaptive1.get_surrogate_type()}")
print(f"Uses interactions: {adaptive1.uses_interactions()}")

In [ ]:
# Test on data without interactions
no_int_data = create_synthetic_interaction_dataset(
    n_samples=2000,
    interaction_strength=0.0  # No interactions
)
blackbox2 = train_model_for_dataset(no_int_data, model_type='xgboost')

print("Test 2: Dataset with NO interactions")
adaptive2 = AdaptiveSurrogate(fidelity_threshold=0.85)
adaptive2.fit(blackbox2.model, no_int_data['X_train'])

print(f"\nDecision: {adaptive2.get_surrogate_type()}")
print(f"Uses interactions: {adaptive2.uses_interactions()}")

## 6. Summary

### Key Contributions

1. **Gap Identification**: We identified that additive surrogates cannot capture feature interactions

2. **GA²M Surrogate**: We implemented interaction-aware surrogates using EBM with pairwise terms

3. **Extended Shapley Formula**: We fairly allocate interaction contributions between features

4. **Adaptive Strategy**: We automatically select the best surrogate based on fidelity

### Results

- **Improved Accuracy**: Higher correlation with Exact SHAP on interaction-heavy data
- **Maintained Speed**: Still much faster than Exact SHAP
- **No Harm**: Doesn't degrade performance on non-interaction data

In [ ]:
print("="*60)
print("EXTENSION SUMMARY")
print("="*60)
print(f"\nOn interaction-heavy data:")
print(f"  Additive surrogate R²: {add_r2:.4f}")
print(f"  GA²M surrogate R²:     {ga2m_r2:.4f}")
print(f"  Fidelity improvement:  {ga2m_r2 - add_r2:.4f}")
print(f"\n  Original InstaSHAP correlation: {original_r:.4f}")
print(f"  Enhanced InstaSHAP correlation: {enhanced_r:.4f}")
print(f"  Accuracy improvement: {enhanced_r - original_r:.4f}")
print("\nExtension successfully addresses the identified gap!")